In [1]:
from ultralytics import YOLO
import torch
import os
import cv2

In [2]:
model = YOLO("yolo26n.pt")
print(model.names)

{0: 'blue', 1: 'orange', 2: 'yellow', 3: 'unknown'}


In [ ]:
img = cv2.imread("test-cone-image.jpg")
results = model(img,conf=0.5)

detections = []

for r in results:
    vis = r.plot()
    cv2.imshow("YOLO detections", vis)

    key = cv2.waitKey(0)   # waits until you press a key
    if key == 27:          # Esc key
        break

    for b in r.boxes:
        cls_id = int(b.cls[0])
        conf = float(b.conf[0])
        x1, y1, x2, y2 = b.xyxy[0].tolist()

        print("class:", cls_id)
        print("label:", model.names[cls_id])
        print("conf:", conf)
        print("bbox:", [x1, y1, x2, y2])
        print()

        detections.append({"class": cls_id, "label": model.names[cls_id], "conf": conf,"bbox": [x1, y1, x2, y2]})
    
cv2.destroyAllWindows()


0: 384x640 1 orange, 5 yellows, 285.6ms
Speed: 11.5ms preprocess, 285.6ms inference, 20.5ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
def convert_centroids(centroids, fx, fy, cx, cy, z0, offset=None):
    """
    CONVERT CENTROID LOCATIONS FROM LIDAR FRAME TO PIXEL LOCATION 


    centroids: (N, 2) tensor with columns [x, y] in LiDAR frame
               where x is forward and y is right
    z0: fixed LiDAR-frame z value used to lift 2D centroids into 3D
    """

    # Add fixed z-coordinate
    new_col = torch.full(
        (centroids.shape[0], 1),
        z0,
        dtype=centroids.dtype,
        device=centroids.device
    )
    centroids_3d = torch.cat([centroids, new_col], dim=1)   # (N, 3)

    # LiDAR -> camera rotation
    rotation = torch.tensor([
        [0,  1,  0],
        [0,  0, -1],
        [1,  0,  0]
    ], dtype=centroids.dtype, device=centroids.device)

    # Translation in camera frame
    if offset is None:
        offset = torch.zeros(3, dtype=centroids.dtype, device=centroids.device)

    # Transform into camera frame
    centroids_cam = centroids_3d @ rotation.T + offset   # (N, 3)

    # Split coordinates
    X = centroids_cam[:, 0]
    Y = centroids_cam[:, 1]
    Z = centroids_cam[:, 2]

    # Avoid divide-by-zero / behind-camera points
    valid = Z > 0

    u = torch.full_like(Z, float("nan"))
    v = torch.full_like(Z, float("nan"))

    u[valid] = fx * (X[valid] / Z[valid]) + cx
    v[valid] = fy * (Y[valid] / Z[valid]) + cy

    uv = torch.stack([u, v], dim=1)   # (N, 2)

    return uv, valid


0: 384x640 1 orange, 5 yellows, 32.1ms
Speed: 874.4ms preprocess, 32.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
def check_centroid(uv, detections): 
    # check if a centroid is within any bounding boxs and return the info for that centroid

    u, v = float(uv[0]), float(uv[1])

    candidates = []

    for det in detections:
        x1, y1, x2, y2 = det["bbox"]

        if x1 <= u <= x2 and y1 <= v <= y2:
            cx = 0.5 * (x1 + x2)
            cy = 0.5 * (y1 + y2)
            dist = math.hypot(u - cx, v - cy)
            candidates.append((dist, -det["conf"], det))

    if not candidates:
        return {"colour": "unknown", "conf": 0.0, "bbox": None}

    candidates.sort(key=lambda x: (x[0], x[1]))
    best = candidates[0][2]

    return {
        "colour": best["class_name"],
        "conf": best["conf"],
        "bbox": best["bbox"]
    }
    
uv_points, valid = convert_centroids(centroids, fx, fy, cx, cy, z0, offset=None)

colour_to_id = {
    "blue": 0,
    "yellow": 1,
    "orange": 2,
    "unknown": -1
}

def fuse_centroids_with_colour(centroids, uv_points, detections):
    """
    centroids: (N, 2) tensor of LiDAR centroids [x, y]
    uv_points: (N, 2) tensor of projected image points [u, v]
    detections: YOLO detections list

    returns:
        fused: (N, 3) tensor [x, y, colour_id]
    """
    fused = torch.zeros((centroids.shape[0], 3), dtype=torch.float32, device=centroids.device)

    for i, uv in enumerate(uv_points):
        result = check_centroid(uv, detections)
        colour_id = colour_to_id.get(result["colour"], -1)

        fused[i, 0] = centroids[i, 0]   # original x
        fused[i, 1] = centroids[i, 1]   # original y
        fused[i, 2] = colour_id         # assigned class

    return fused